# Módulo 1 · ADK Code-First + A2A
**Caso:** Supervisor de Finanzas que delega a un Especialista en Presupuestos, manteniendo el estado de la conversación.

## Parte 0 · Preparación (10 min)

In [1]:
import os, sys, subprocess, asyncio, time
from pathlib import Path

# 1) Ubica la raíz del proyecto y prepara los imports
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "config.py").exists())
for ruta in (RAIZ, RAIZ / "modulo1_adk_a2a" / "src"):
    sys.path.insert(0, str(ruta))
os.chdir(RAIZ)

# 2) Variables de entorno (.env) y datos de ejemplo
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")
if not (RAIZ / "data" / "finanzas.db").exists():
    subprocess.run([sys.executable, "data/seed_data.py"], check=True)

import logging, warnings
logging.getLogger("google_adk").setLevel(logging.ERROR)     # menos ruido de logs de ADK
warnings.filterwarnings("ignore", category=UserWarning)     # avisos EXPERIMENTAL de ADK (RemoteA2aAgent, etc.)

import google.adk
print("RAIZ:", RAIZ)
print("google-adk:", google.adk.__version__)

RAIZ: c:\Users\Sofia Donlucas\Documents\Visual\GeminiAgentsBI
google-adk: 2.11.0


### Modelo: Gemini en modo enterprise

In [2]:
from finanzas_agents.eventos import preguntar, estado 
from shared.config import MODEL_NAME

# Credenciales de gcloud.
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

proyecto = os.getenv("GOOGLE_CLOUD_PROJECT", "")
if not proyecto or proyecto == "tu-proyecto":
    raise RuntimeError("Falta GOOGLE_CLOUD_PROJECT en tu .env: pon el ID de tu proyecto (no 'tu-proyecto').")

import google.auth
try:
    google.auth.default() 
except google.auth.exceptions.DefaultCredentialsError:
    raise RuntimeError("No hay credenciales. En una terminal ejecuta: gcloud auth application-default login")

MODELO = MODEL_NAME
os.environ["MODEL_NAME"] = MODEL_NAME
print("Proyecto:", proyecto, "| Región:", os.environ["GOOGLE_CLOUD_LOCATION"], "| Modelo:", MODELO)

Proyecto: ninth-stock-509413-g4 | Región: us-central1 | Modelo: gemini-2.5-flash


## Parte 1 · Herramienta tipada y contratos 
Una **herramienta** es una función Python con tipos y *docstring*: el modelo lee el docstring para decidir cuándo y cómo llamarla.

In [3]:
from finanzas_agents.tools import consultar_presupuesto
import inspect
print(inspect.signature(consultar_presupuesto))
print(inspect.getdoc(consultar_presupuesto)[:200], "...\n")

r = consultar_presupuesto("Norte", "2026-Q3")
print(r["status"], r["division"], r["trimestre"], "→ desviación", r["desviacion_pct"], "%")
for l in r["lineas"]:
    print(f"  {l['categoria']:12} presup {l['presupuestado']:>9,.0f}  ejec {l['ejecutado']:>9,.0f}")

(division: str, trimestre: str, categoria: str = '') -> dict
Consulta presupuesto vs ejecutado de una división y trimestre.

Args:
    division: Norte, Sur, Centro u Occidente.
    trimestre: formato AAAA-Qn, por ejemplo 2026-Q3.
    categoria: opcional (Viatic ...

ok Norte 2026-Q3 → desviación 25.3 %
  Viaticos     presup    47,000  ejec    68,150
  Consultoria  presup    78,000  ejec   124,800
  Tecnologia   presup    55,000  ejec    57,750
  Marketing    presup    56,000  ejec    58,800
  Servicios    presup    68,000  ejec    71,400


**Ejercicio 1.** Consulta la división **Sur**, trimestre **2026-Q1**, solo la categoría **Tecnologia**, e imprime su desviación.

In [4]:
r1 = consultar_presupuesto("Sur", "2026-Q1", "Tecnologia")
print(r1["status"], r1["lineas"], "→ desviación", r1["desviacion_pct"], "%")

ok [{'categoria': 'Tecnologia', 'presupuestado': 55000.0, 'ejecutado': 48541.37}] → desviación -11.74 %


**Contratos Pydantic:** la validación estricta ocurre *antes* del LLM. Una petición inválida se rechaza por código, no por criterio del modelo.

In [5]:
from pydantic import ValidationError
from finanzas_agents.contracts import BudgetRequest

print(BudgetRequest(division="Norte", trimestre="2026-Q3"))
try:
    BudgetRequest(division="Marte", trimestre="2026-Q3")
except ValidationError as e:
    print("❌ rechazado:", e.errors()[0]["msg"])

division='Norte' trimestre='2026-Q3' categoria=None
❌ rechazado: Input should be 'Norte', 'Sur', 'Centro' or 'Occidente'


**Ejercicio 2.** Demuestra que el trimestre **`2026-Q5`** también se rechaza e imprime el mensaje de error.

In [6]:
try:
    BudgetRequest(division="Norte", trimestre="2026-Q5")
except ValidationError as e:
    print("❌ rechazado:", e.errors()[0]["msg"])

❌ rechazado: String should match pattern '^\d{4}-Q[1-4]$'


## Parte 2 · Tu primer agente
Un agente = **modelo + instrucción + herramientas**. `description` = *qué es* (la lee el router); `instruction` = *cómo* cumple su objetivo.

In [7]:
from google.adk.agents import Agent

presupuestos_agent = Agent(
    name="especialista_presupuestos",          # sin espacios
    model=MODELO,
    description=("Especialista en presupuestos: consulta presupuesto vs ejecutado por "
                 "división, trimestre y categoría, y calcula desviaciones."),
    instruction=("Eres un analista de presupuestos. Usa SIEMPRE la herramienta consultar_presupuesto; "
                 "nunca inventes cifras. Responde con totales, desviación % y las 2 categorías con "
                 "mayor exceso. Si falta división o trimestre, pídelos. "
                 "Reporta las cifras EXACTAMENTE como las devuelve la herramienta: mismas unidades, "
                 "sin multiplicarlas, convertirlas ni redondearlas."),
    tools=[consultar_presupuesto],
)

In [8]:
# Ejecuta el agente. Observa los eventos: 🔧 llamada a herramienta → 📥 resultado → 🤖 respuesta final
await preguntar(presupuestos_agent, "¿Cómo va la división Norte en 2026-Q3?", session_id="p2")

👤 ¿Cómo va la división Norte en 2026-Q3?
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'division': 'Norte', 'trimestre': '2026-Q3'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'status': 'ok', 'division': 'Norte', 'trimestre': '2026-Q3', 'lineas': [{'categoria': 'Viaticos', 'presupuest…
🤖 [especialista_presupuestos] La división Norte en 2026-Q3 tiene un presupuesto total de 304000, un ejecutado total de 380900, y una desviación del 25.3%. Las dos categorías con mayor exceso son Consultoria (124800 ejecutado vs 78000 presupuestado) y Viaticos (68150 ejecutado vs 47000 presupuestado).


**Para discutir (2 min):**
1. ¿Qué evento muestra que el agente usó la herramienta en vez de "inventar"?
2. ¿Qué pasa si la `instruction` no dice "usa SIEMPRE la herramienta"? (con Gemini real, pruébalo)
3. ¿Dónde viven las cifras: en el prompt o en la base de datos?

## Parte 3 · Supervisor y delegación
El **Supervisor** no consulta presupuestos: decide a quién delegar usando las `description` de sus `sub_agents` (hand-off jerárquico).

In [9]:
supervisor = Agent(
    name="supervisor_finanzas",
    model=MODELO,
    description="Orquestador financiero que delega a especialistas.",
    instruction=("Eres el Supervisor de Finanzas. Analiza la solicitud y delega: preguntas de "
                 "presupuesto/ejecución -> especialista_presupuestos. Conserva el contexto de la "
                 "conversación (división y trimestre ya mencionados). Resume con cifras exactas."),
    sub_agents=[presupuestos_agent],
)

In [10]:
await preguntar(supervisor, "¿Cómo va la división Norte en 2026-Q3?", session_id="s2")

👤 ¿Cómo va la división Norte en 2026-Q3?
   🔧 [supervisor_finanzas] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_finanzas] resultado de transfer_to_agent: {'result': None}
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'trimestre': '2026-Q3', 'division': 'Norte'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'status': 'ok', 'division': 'Norte', 'trimestre': '2026-Q3', 'lineas': [{'categoria': 'Viaticos', 'presupuest…
🤖 [especialista_presupuestos] La división Norte en 2026-Q3 tiene un presupuesto total de 304000, con un ejecutado de 380900, lo que representa una desviación del 25.3%.

Las 2 categorías con mayor exceso son:
1. Consultoria: 46800
2. Viaticos: 21150


**Observa:** primero `🔧 [supervisor_finanzas] llama transfer_to_agent(...)` y luego el especialista llama a su herramienta.
Eso es la **delegación**. Además, `adk web` muestra este mismo flujo (con un grafo) en su pestaña *Events*. Ábrelo en **una terminal aparte** (no en el notebook):
```
. .\env.ps1
adk web --port 8501 modulo1_adk_a2a/src
```
Luego entra a http://127.0.0.1:8501 y elige `finanzas_agents`.

> Si repites la celda de arriba con el **mismo `session_id`**, no verás de nuevo los eventos (la conversación continúa). Usa un id nuevo (`s2`, `s3`...).

## Parte 4 · Estado de la conversación
La memoria de corto plazo vive en la **sesión**, no en el prompt. Misma `session_id` → misma conversación.

In [11]:
# Seguimiento en la MISMA sesión: no repetimos la división
await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="s2")

👤 ¿Y en 2026-Q2?
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'division': 'Norte', 'trimestre': '2026-Q2'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'status': 'ok', 'division': 'Norte', 'trimestre': '2026-Q2', 'lineas': [{'categoria': 'Viaticos', 'presupuest…
🤖 [especialista_presupuestos] La división Norte en 2026-Q2 tiene un presupuesto total de 318000, con un ejecutado de 294364.01, lo que representa una desviación del -7.43%.

No hay categorías con exceso, ya que la desviación total es negativa.


In [12]:
# Otra sesión = otra conversación: no hay contexto previo
await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="s3")

👤 ¿Y en 2026-Q2?
   🔧 [supervisor_finanzas] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_finanzas] resultado de transfer_to_agent: {'result': None}
🤖 [especialista_presupuestos] Para 2026-Q2, ¿de qué división necesitas el presupuesto?


**Pregunta de examen:** ¿por qué la segunda respuesta pide la división? ¿Qué componente guarda la conversación?
*(Respuesta: el `SessionService`; aquí `InMemorySessionService`. En producción: base de datos o Agent Runtime Sessions.)*

**Experimento (con Gemini real):** cambia la pregunta a solo **"¿Y en Q2?"** (sin año) y repítela 3 veces en sesiones nuevas.
En mis pruebas, a veces deduce 2026 y a veces pregunta *"¿de qué año?"*. Es **no determinismo**: la misma entrada puede dar caminos distintos. Por eso en las pruebas de agentes se miden **tasas de éxito**, no un solo intento (Módulo 5).

## Parte 5 · A2A: del handshake al servidor real

### 5.1 Protocolo sin red ni LLM

In [13]:
from finanzas_agents.a2a_local import Supervisor, BudgetWorker

sup_local = Supervisor(workers={"especialista_presupuestos": BudgetWorker()})
print("handshake →", sup_local.handshake("especialista_presupuestos"))
print(sup_local.ask("sesion1", "¿Cómo va la división Norte en Q3?"))
print(sup_local.ask("sesion1", "¿Y en Q2?"))
print("estado →", sup_local.sessions)

handshake → AgentCard(name='especialista_presupuestos', skills=['presupuesto'])
Norte 2026-Q3: presupuestado 304,000, ejecutado 380,900 (+25.3%).
Norte 2026-Q2: presupuestado 318,000, ejecutado 294,364 (-7.4%).
estado → {'sesion1': {'division': 'Norte', 'trimestre': '2026-Q2', 'anio': '2026', 'ultima_desviacion': -7.43}}


### 5.2 A2A real por HTTP
Levantamos el especialista como **servidor A2A** (puerto 8001) y el supervisor lo consume como agente remoto.

In [14]:
import httpx

entorno = {**os.environ,
           "PYTHONPATH": os.pathsep.join([str(RAIZ), str(RAIZ / "modulo1_adk_a2a" / "src")])}
servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "finanzas_agents.a2a_server:a2a_app", "--port", "8001"],
    env=entorno, cwd=RAIZ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

URL_CARD = "http://localhost:8001/.well-known/agent-card.json"
card = None
for _ in range(60):
    try:
        card = httpx.get(URL_CARD, timeout=2).json()
        break
    except Exception:
        time.sleep(0.5)
assert card, "El servidor A2A no arrancó: revisa el puerto 8001"
print("Agent card →", card["name"], "|", card["description"][:70], "...")

Agent card → especialista_presupuestos | Especialista en presupuestos: consulta presupuesto vs ejecutado por di ...


In [15]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

especialista_remoto = RemoteA2aAgent(
    name="especialista_presupuestos",
    description="Especialista de presupuestos expuesto vía A2A.",
    agent_card=URL_CARD,
)
supervisor_remoto = Agent(
    name="supervisor_remoto",
    model=MODELO,
    description="Supervisor que delega por red.",
    instruction="Eres el Supervisor de Finanzas. Delega las consultas de presupuesto al especialista remoto.",
    sub_agents=[especialista_remoto],
)

In [16]:
await preguntar(supervisor_remoto, "¿Cómo va la división Norte en 2026-Q3?", session_id="a2a1")

👤 ¿Cómo va la división Norte en 2026-Q3?
   🔧 [supervisor_remoto] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_remoto] resultado de transfer_to_agent: {'result': None}
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'trimestre': '2026-Q3', 'division': 'Norte'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'trimestre': '2026-Q3', 'total_presupuestado': 304000.0, 'division': 'Norte', 'lineas': [{'presupuestado': 47…
🤖 [especialista_presupuestos] La división Norte en el 2026-Q3 tiene un total ejecutado de 380900 y un total presupuestado de 304000, lo que representa una desviación del 25.3%.

Las 2 categorías con mayor exceso son:
* Consultoria: 124800 ejecutado vs 78000 presupuestado
* Viaticos: 68150 ejecutado vs 47000 presupuestado


### 5.3 Experimento: ¿el estado también viaja por A2A?
Ya viste que la sesión recuerda la conversación con el supervisor **local** (Parte 4). ¿Pasa lo mismo cuando el especialista es **remoto**? Ejecuta las dos celdas siguientes **antes de cerrar el servidor** (la celda de `servidor.terminate()`): la primera usa la **misma sesión** (`a2a1`) y la segunda una **sesión nueva** como control.
Observa: ¿responde sobre Norte sin que repitas la división, o te la vuelve a pedir? Anota lo que veas.

In [17]:
# Seguimiento por A2A: MISMA sesión que la pregunta anterior (a2a1)
await preguntar(supervisor_remoto, "¿Y en 2026-Q2?", session_id="a2a1")

👤 ¿Y en 2026-Q2?
   🔧 [supervisor_remoto] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_remoto] resultado de transfer_to_agent: {'result': None}
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'trimestre': '2026-Q2', 'division': 'Norte'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'trimestre': '2026-Q2', 'total_presupuestado': 318000.0, 'division': 'Norte', 'lineas': [{'presupuestado': 82…
🤖 [especialista_presupuestos] La división Norte en el 2026-Q2 tiene un total ejecutado de 294364.01 y un total presupuestado de 318000, lo que representa una desviación del -7.43%.

Las 2 categorías con mayor exceso son:
* Consultoria: 52184.76 ejecutado vs 54000 presupuestado
* Tecnologia: 50955.5 ejecutado vs 53000 presupuestado


In [18]:
# Control: sesión NUEVA (no debería tener contexto y pedir la división)
await preguntar(supervisor_remoto, "¿Y en 2026-Q2?", session_id="a2a2")

👤 ¿Y en 2026-Q2?
   🔧 [supervisor_remoto] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_remoto] resultado de transfer_to_agent: {'result': None}
🤖 [especialista_presupuestos] Por favor, especifica la división (Norte, Sur, Centro u Occidente).


### 5.4 Reto opcional: router con **dos** especialistas remotos (10 min)
Hasta aquí el supervisor tenía **un** especialista remoto. Ahora publicamos **Tesorería** como segundo servidor A2A (puerto **8002**) y el supervisor decide a cuál transferir según el dominio de la pregunta.

> Recuerda: un agente remoto es una **caja negra** para el supervisor. Quien decide a quién pasar el control es **el supervisor**, no los especialistas entre sí (a diferencia del hand-off **local** de la Parte 7).
> Ejecútalo **antes de cerrar el servidor de la 5.2** (la celda de `servidor.terminate()`).

In [19]:
# Segundo servidor A2A: Tesorería en el puerto 8002 (el de Presupuestos sigue en el 8001)
servidor_tes = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "finanzas_agents.a2a_server:a2a_app_tesoreria", "--port", "8002"],
    env=entorno, cwd=RAIZ, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

URL_CARD_TES = "http://localhost:8002/.well-known/agent-card.json"
card_tes = None
for _ in range(60):
    try:
        card_tes = httpx.get(URL_CARD_TES, timeout=2).json()
        break
    except Exception:
        time.sleep(0.5)
assert card_tes, "El servidor de Tesorería no arrancó: revisa el puerto 8002"
print("Agent card →", card_tes["name"], "|", card_tes["description"][:70])

Agent card → especialista_tesoreria | Flujo de caja, liquidez y pagos.


In [20]:
# Un RemoteA2aAgent por especialista (un agente solo puede tener UN padre: no reutilices los de la 5.2)
presupuestos_rem = RemoteA2aAgent(name="especialista_presupuestos",
                                  description="Presupuesto vs ejecutado por división y trimestre.",
                                  agent_card=URL_CARD)
tesoreria_rem = RemoteA2aAgent(name="especialista_tesoreria",
                               description="Flujo de caja, liquidez y pagos.",
                               agent_card=URL_CARD_TES)

supervisor_router = Agent(
    name="supervisor_router",
    model=MODELO,
    description="Supervisor que enruta por red a dos especialistas remotos.",
    instruction=("Eres el Supervisor de Finanzas. Presupuesto/ejecución -> especialista_presupuestos; "
                 "liquidez/caja/pagos -> especialista_tesoreria. Delega siempre en un especialista."),
    sub_agents=[presupuestos_rem, tesoreria_rem],
)

# Dos preguntas de dominios distintos en la MISMA sesión
await preguntar(supervisor_router, "¿Cuál es la posición de liquidez?", session_id="r1")
await preguntar(supervisor_router, "¿y el presupuesto de Norte en 2026-Q3?", session_id="r1")

👤 ¿Cuál es la posición de liquidez?
   🔧 [supervisor_router] llama transfer_to_agent({'agent_name': 'especialista_tesoreria'})
   📥 [supervisor_router] resultado de transfer_to_agent: {'result': None}
🤖 [especialista_tesoreria] No tengo acceso a los datos de la empresa para darte una cifra exacta sobre la posición de liquidez. Para determinarla, necesitaría información sobre el saldo de caja y equivalentes, las inversiones a corto plazo, las cuentas por cobrar a corto plazo, los pasivos circulantes (cuentas por pagar, deuda a corto plazo), y cualquier línea de crédito disponible.
👤 ¿y el presupuesto de Norte en 2026-Q3?
   🔧 [supervisor_router] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [supervisor_router] resultado de transfer_to_agent: {'result': None}
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'trimestre': '2026-Q3', 'division': 'Norte'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'trimestre': '2026-Q3', 't

In [21]:
# Cierra el servidor de Tesorería
servidor_tes.terminate()
print("servidor de Tesorería detenido")

servidor de Tesorería detenido


In [22]:
# Cierra el servidor cuando termines la Parte 5 (o déjalo si vas a repetir la celda anterior)
servidor.terminate()
print("servidor A2A detenido")

servidor A2A detenido


**Diferencia clave:** `sub_agents` locales = mismo proceso; **A2A** = otro servicio (otro equipo, otro lenguaje, otra nube) con un contrato publicado (la *agent card*).
> `RemoteA2aAgent` está marcado como **experimental** en ADK: si cambia su firma, consulta https://adk.dev/a2a/quickstart-consuming/
> En A2A, quien decide a quién pasar el control es **el supervisor**: un especialista remoto es una caja negra y no ve el árbol (en el hand-off **local** de la Parte 7, en cambio, Tesorería transfiere directo a Presupuestos).

## Parte 6 · Workflows y `output_key`
`SequentialAgent`, `ParallelAgent` y `LoopAgent` orquestan **sin que un LLM decida el orden**. `output_key` guarda la respuesta de un agente en el **estado** para el siguiente.

> ⚠️ **Aviso (ADK 2.x):** estas clases funcionan pero están **marcadas como obsoletas** a favor de `Workflow` (grafos de ejecución: mezclan código determinista y razonamiento del LLM). Hoy `Workflow` aún no puede usarse como sub-agente de un `LlmAgent`. Enseñamos los 3 tipos porque son el vocabulario del curso y de tus notas; revisa https://adk.dev para la ruta de migración.

In [23]:
from google.adk.agents import SequentialAgent

analista = Agent(name="analista_presupuesto", model=MODELO,
                 description="Consulta presupuesto vs ejecutado.",
                 instruction=("Usa consultar_presupuesto para la división y trimestre pedidos. Resume totales y desviación %. "
                              "Reporta las cifras EXACTAMENTE como las devuelve la herramienta: mismas unidades, "
                              "sin multiplicarlas, convertirlas ni redondearlas."),
                 tools=[consultar_presupuesto], output_key="analisis_presupuesto")
redactor = Agent(name="redactor_ejecutivo", model=MODELO,
                 description="Redacta resúmenes ejecutivos.",
                 instruction=("Redacta un resumen ejecutivo de 3 líneas basado SOLO en este análisis: {analisis_presupuesto}. "
                              "Copia las cifras tal cual aparecen en el análisis, sin cambiar sus unidades."),
                 output_key="resumen_ejecutivo")
pipeline = SequentialAgent(name="pipeline_presupuesto", sub_agents=[analista, redactor])

await preguntar(pipeline, "Norte 2026-Q3", session_id="w2")
print("\nEstado de la sesión →", list((await estado(pipeline, "w2")).keys()))

👤 Norte 2026-Q3


C:\Users\Sofia Donlucas\AppData\Local\Temp\ipykernel_13980\210598825.py:14: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  pipeline = SequentialAgent(name="pipeline_presupuesto", sub_agents=[analista, redactor])


   🔧 [analista_presupuesto] llama consultar_presupuesto({'trimestre': '2026-Q3', 'division': 'Norte'})
   📥 [analista_presupuesto] resultado de consultar_presupuesto: {'status': 'ok', 'division': 'Norte', 'trimestre': '2026-Q3', 'lineas': [{'categoria': 'Viaticos', 'presupuest…
🤖 [analista_presupuesto] El total presupuestado para Norte en 2026-Q3 fue de 304000 y el total ejecutado fue de 380900. La desviación porcentual es del 25.3%.
🤖 [redactor_ejecutivo] Para Norte en 2026-Q3, el total presupuestado fue de 304000.
El total ejecutado ascendió a 380900.
Esto representa una desviación porcentual del 25.3%.

Estado de la sesión → ['analisis_presupuesto', 'resumen_ejecutivo']


**Ejercicio 3.** Crea un `LoopAgent` llamado `refinador` que repita **3 veces** un agente `revisor` (sin `model`, sin `instruction` en el LoopAgent) e imprime su `max_iterations`.

In [24]:
from google.adk.agents import LoopAgent

revisor = Agent(name="revisor", model=MODELO, description="Mejora el resumen.",
                instruction="Mejora la claridad de: {resumen_ejecutivo}. Mantén las cifras exactas.",
                output_key="resumen_ejecutivo")
refinador = LoopAgent(name="refinador", sub_agents=[revisor], max_iterations=3)
print(refinador.max_iterations)

3


C:\Users\Sofia Donlucas\AppData\Local\Temp\ipykernel_13980\3259807261.py:6: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  refinador = LoopAgent(name="refinador", sub_agents=[revisor], max_iterations=3)


## Parte 7 · Reto: hand-off jerárquico a Tesorería (10 min)
Agrega un segundo especialista y comprueba que el control pasa de uno a otro según el dominio de la consulta.

> ⚠️ **Regla de ADK:** un agente solo puede tener **un padre**. `presupuestos_agent` ya cuelga de `supervisor`; para reutilizarlo bajo otro supervisor usa **`presupuestos_agent.clone()`** (copia sin padre).

In [25]:
tesoreria_agent = Agent(
    name="especialista_tesoreria", model=MODELO,
    description="Flujo de caja, liquidez y pagos.",
    instruction=("Eres el especialista de Tesorería (flujo de caja, liquidez y pagos). "
                 "No tienes herramientas ni acceso a datos de la empresa: no des cifras ni inventes datos; "
                 "si te piden un dato, explica qué información necesitarías. "
                 "Si te preguntan por presupuesto o ejecución, devuelve el control."))

supervisor_v2 = Agent(
    name="supervisor_finanzas_v2", model=MODELO,
    description="Orquestador financiero con dos especialistas.",
    instruction=("Eres el Supervisor de Finanzas. Presupuesto/ejecución -> especialista_presupuestos; "
                 "liquidez/caja/pagos -> especialista_tesoreria."),
    sub_agents=[presupuestos_agent.clone(), tesoreria_agent])

await preguntar(supervisor_v2, "¿Cuál es la posición de liquidez?", session_id="h2")
await preguntar(supervisor_v2, "¿y el presupuesto de Norte en 2026-Q3?", session_id="h2")

👤 ¿Cuál es la posición de liquidez?
   🔧 [supervisor_finanzas_v2] llama transfer_to_agent({'agent_name': 'especialista_tesoreria'})
   📥 [supervisor_finanzas_v2] resultado de transfer_to_agent: {'result': None}
🤖 [especialista_tesoreria] Para poder determinar la posición de liquidez, necesitaría acceso a la información financiera de la empresa, incluyendo los estados de flujo de efectivo, el balance general (especialmente los activos líquidos y pasivos a corto plazo) y los pronósticos de ingresos y gastos. Sin esta información, no puedo proporcionar una cifra o evaluación específica.
👤 ¿y el presupuesto de Norte en 2026-Q3?
   🔧 [especialista_tesoreria] llama transfer_to_agent({'agent_name': 'especialista_presupuestos'})
   📥 [especialista_tesoreria] resultado de transfer_to_agent: {'result': None}
   🔧 [especialista_presupuestos] llama consultar_presupuesto({'division': 'Norte', 'trimestre': '2026-Q3'})
   📥 [especialista_presupuestos] resultado de consultar_presupuesto: {'status': 'o

## Parte 8 · Autoevaluación del entregable (10 min)
Ejecuta la celda: debes ver ✅ en todo. Si algo marca ❌, vuelve a la parte indicada.

In [26]:
def check(nombre, cond, parte):
    print(("✅" if cond else "❌ (revisa Parte %s)" % parte), nombre)
    return cond

resultados = []

# Contratos
try:
    BudgetRequest(division="Marte", trimestre="2026-Q3"); rechaza = False
except ValidationError:
    rechaza = True
resultados.append(check("Un contrato inválido se rechaza por código, no por el LLM", rechaza, 1))

# Herramienta
t = consultar_presupuesto("Norte", "2026-Q3")
resultados.append(check("La herramienta devuelve Norte 2026-Q3 con exceso > 15 %", t["status"] == "ok" and t["desviacion_pct"] > 15, 1))

# Delegación + estado (sesión nueva y limpia)
# Gemini puede escribir las cifras como 25.3 / 25,3 y 380,900 / 380.900 / 380900: aceptamos todas.
r1 = await preguntar(supervisor, "¿Cómo va la división Norte en 2026-Q3?", session_id="eval", verbose=False)
resultados.append(check("El supervisor delega y responde con la cifra (25.3 % / 380,900)",
                        any(x in r1 for x in ("25.3", "25,3", "380,900", "380.900", "380900")), 3))
r2 = await preguntar(supervisor, "¿Y en 2026-Q2?", session_id="eval", verbose=False)
# Gemini no es determinista: lo que NO debe pasar es que vuelva a pedir la DIVISIÓN (el contexto se perdió).
pide_division = "divisi" in r2.lower() and "norte" not in r2.lower()
resultados.append(check("El seguimiento conserva la división (no la vuelve a pedir)", bool(r2) and not pide_division, 4))

# A2A
resultados.append(check("La agent card A2A se publicó", bool(card) and card.get("name") == "especialista_presupuestos", 5))

print("\n🎓 ENTREGABLE LISTO" if all(resultados) else "\n⚠️ Falta algo: revisa los ❌")

✅ Un contrato inválido se rechaza por código, no por el LLM
✅ La herramienta devuelve Norte 2026-Q3 con exceso > 15 %
✅ El supervisor delega y responde con la cifra (25.3 % / 380,900)
✅ El seguimiento conserva la división (no la vuelve a pedir)
✅ La agent card A2A se publicó

🎓 ENTREGABLE LISTO


### Siguiente paso: de tu notebook a un proyecto
Lo que construiste en estas celdas ya existe como **proyecto real** en `modulo1_adk_a2a/src/finanzas_agents/`:
- Compara tu `presupuestos_agent` y tu `tesoreria_agent` con `agent.py`. Fíjate en que allí el supervisor se crea **una sola vez con los dos especialistas**, así que no hace falta `clone()`.
- Lee el `README.md` de esa carpeta: mapa de archivos, cómo ejecutarlo con `adk web` y cómo ampliarlo.
- Esa carpeta es la **base que vas a ampliar** en los siguientes módulos (RAG, SQL, ML) y en tu entregable final. El notebook fue la guía; el proyecto es lo que te llevas.

### Entrega y code review (tutoría)
- Sube tu notebook y el repo de la célula.
- En el code review mostraremos: `description` de cada agente, contratos Pydantic, manejo de errores como dato y ausencia de secretos.

**Siguiente módulo:** RAG financiero con reranking (K=20 → N=3).